In [2]:
!pip install --upgrade openai


[notice] A new release of pip available: 22.3.1 -> 24.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from openai import OpenAI
import os
from tqdm import tqdm

In [ ]:
os.environ["OPENAI_API_KEY"] = " "

In [23]:
from openai import OpenAI # Import the OpenAI package
import os # Import the os package

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [33]:
import pandas as pd

df = pd.read_csv("../../manual_analysis/samples/python.csv")
df.columns

Index(['top-candidate-scorer-id', 'candidate-relevance-score',
       'RelevantAnswerDetail', 'BugReport', 'AcceptedAnswer', 'Question',
       'MistralAnswer', 'GoldBugReport', 'BLEU_SCORE', 'SEMANTIC_SIMILARITY',
       'METEOR_SCORE', 'ROUGE_SCORE', 'Embedding_Similarity_Score'],
      dtype='object')

In [34]:
for index, row in tqdm(df.iterrows(), total=len(df)):
    question = row.Question
    bug_report = row['GoldBugReport']
    relevant_info = row['RelevantAnswerDetail']
    prompt = """Answer Question on the bug report based on the relevant information.
    Here is a bug report which has incomplete information.
    ## Bug Report - \n""" + str(bug_report) +  """ There is a follow up question asking for missing information.
    \n Here is some relevant information from previous bug report.\n"""+ str(relevant_info) + """\n
    Can you answer the question below based on the bug report. 
    Here is the question.\n 
    ## Question- \n""" + str(question) + "\n ## Answer : \n "
    
    completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": str(prompt)}])
    
    df.at[index, 'ChatGPTAnswer'] = completion.choices[0].message.content
print(df.head())
df.to_csv("../../results/chatgpt/python.csv", index=False)

100%|██████████| 49/49 [01:20<00:00,  1.64s/it]

  top-candidate-scorer-id  candidate-relevance-score  \
0                 53152-2                3997.360097   
1                  5023-1                1497.777252   
2                 70273-1                2168.915830   
3                  1815-2                1101.109119   
4                  3971-2                 213.886427   

                                RelevantAnswerDetail  \
0  The machines I rolled back to 2018.3.3-1 are b...   
1  \nI'm sorry, I don't understand what you are r...   
2  Note that setting the `ansible_python_interpre...   
3  xning thanks. I tried to build the dev image l...   
4  \nSorry, I was mistaken.\n\nThis is now offici...   

                                           BugReport  \
0  b'daily highstate fails after 2019.2 upgrade'b...   
1  Eq() gets .as_basic() methodA common idiom is ...   
2  b'Python interpreter discovery not discovering...   
3  Worker thread stuck in die state  Describe the...   
4  remove dependency on pexpect and add suppor